## Imports

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import datetime as datetime

from scipy import stats
from pyhive import presto
from datetime import datetime, timedelta

from sklearn.model_selection import StratifiedShuffleSplit

import warnings
warnings.filterwarnings('ignore')

In [2]:
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 300)

In [6]:
## Connection
connection = presto.connect(
        
        host='presto-gateway.processing.data.production.internal',
#     presto.processing.yoda.run
#     presto-gateway.processing.data.production.internal',
        port=80,
        protocol='http',
        catalog='hive',
        username='manoj.ravirajan@rapido.bike'
)

## Datasets & Parameter

In [7]:
## Parameter 
start_date = '20240530'
end_date = '20240602'
city = 'Delhi'

# Convert date strings to datetime objects
start_dt = datetime.strptime(start_date, '%Y%m%d')
end_dt = datetime.strptime(end_date, '%Y%m%d')
segment_date = end_dt.strftime('%Y-%m-%d')
print(start_date, ' to ' ,end_date)

20240530  to  20240602


### customer_postorderstatus_started_immutable

In [8]:
df_data = pd.DataFrame()

# Loop through each date in the range
current_dt = start_dt
while current_dt <= end_dt:
    # Format the current date as a string in 'YYYYMMDD' format
    current_date_str = current_dt.strftime('%Y%m%d')
            
    data_query = f"""

        with orderLogsFact as (

        select
            yyyymmdd,
            city_name city,
            service_obj_service_name service_name,
            case when SUBSTR(customer_id, -1) in ('1','3','5', '7', '9', 'a','c','e') then 'l2Cancel' else 'l1Cancel' end pageVariant,
            customer_id,
            order_id,
            order_status,
            modified_order_status,
            spd_fraud_flag,
            CASE WHEN modified_order_status in ('OCARA', 'COBRA', 'COBRM') THEN (customer_cancelled_epoch/1000 - order_requested_epoch/1000) END time_spent_in_sec,
            order_requested_epoch,
            coalesce(dropped_epoch, customer_cancelled_epoch) order_ended_epoch
        from
            orders.order_logs_fact
        where 
            yyyymmdd = '{current_date_str}'
            and city_name = '{city}'
            and service_obj_service_name in ('Link', 'Auto', 'Auto NCR', 'Bike Lite', 'Bike Metro', 'CabEconomy')
        ),

        telemetry_ads as (
        SELECT 
            yyyymmdd,
            city,
            service,
            userId,
            pagename,
            CASE 
            WHEN pagename IN ('HomeScreen', 'CaptainSearchScreen') THEN pagename
            WHEN lower(slotname) LIKE '%ontheway%' THEN 'onTheWay'
            WHEN lower(slotname) LIKE '%arrived%' THEN 'arrived'
            WHEN lower(slotname) LIKE '%started%' THEN 'started'
            WHEN ltrim(orderstatus) IS NOT NULL THEN orderstatus
            END screen_slot_name,
            slotname,
            eventName,
            hhmmss,
            epoch

        FROM 
            canonical.iceberg_log_telemetry_ads_impressions_immutable 
        WHERE  
            yyyymmdd = '{current_date_str}'
            and city = '{city}'
            and responseType = 'GAMBanner'
            and pagename in ('PostOrderScreen', 'CaptainSearchScreen')
            and service in ('Link', 'Auto', 'Auto NCR', 'Bike Lite', 'Bike Metro', 'CabEconomy')
            and lower(slotname) NOT LIKE '%-%'
        ),
        
        
        customer_segment AS (
        SELECT 
            taxi_ltr_segment ltr_segment,
            taxi_retention_segments retention_segment,
            taxi_lifetime_stage lifetime_segment,
            customer_id
        FROM
            datasets.iallocator_customer_segments
        WHERE 
            run_date = '{segment_date}'
            AND taxi_recency_segment != 'INACTIVE'
        ),

        merge as (

        select
            a.*,
            coalesce(c.ltr_segment , 'UNKNOWN') ltr_segment,
            coalesce(c.retention_segment , 'UNKNOWN') retention_segment,
            coalesce(c.lifetime_segment , 'UNKNOWN') lifetime_segment,
            b.pagename,
            b.screen_slot_name screen_slot_name,
            b.slotname,
            b.eventName,
            b.yyyymmdd || '-' || b.userId || '-' || b.slotname || '-' || b.hhmmss unique_event_id
        from 
            orderLogsFact a
        left join 
            telemetry_ads b
            on a.yyyymmdd = b.yyyymmdd
            and a.city = b.city
            and a.service_name = b.service
            and a.customer_id = b.userId
            and b.epoch >= a.order_requested_epoch
            and b.epoch <= a.order_ended_epoch
        left join 
            customer_segment c
            on a.customer_id = c.customer_id
        )


        SELECT
            *
        FROM
            merge

    """
    
    # Execute the query and get the result as a DataFrame
    df_day = pd.read_sql(data_query, connection)

    # Concatenate the result with the existing DataFrame
    df_data = pd.concat([df_data, df_day], ignore_index=True)

    # Move to the next date
    current_dt += timedelta(days=1)

df_data.to_csv('df_experiment_user_base1.csv', index=False)

KeyboardInterrupt: 

In [ ]:
df_experiment_user_base = pd.read_csv('df_experiment_user_base1.csv')
df_experiment_user_base.head(5)

In [ ]:
df_data = df_experiment_user_base.copy()

In [ ]:
df_data.head(2)

In [ ]:
df_data\
.groupby(['pageVariant', 'city'])\
.agg(customers = ('customer_id', 'nunique'))\
.sort_values(by=['pageVariant'])\
.reset_index()

In [ ]:
df_data\
.groupby(['pageVariant', 'city', 'service_name'])\
.agg(customers = ('customer_id', 'nunique'))\
.sort_values(by=['service_name', 'pageVariant'])\
.reset_index()

In [ ]:
df_data\
.groupby(['yyyymmdd', 'pageVariant'])\
.agg(customers = ('customer_id', 'nunique'),
     gross_orders = ('order_id', 'nunique'),
     net_orders = ('order_id', lambda x:x[(df_data['order_status'] == 'dropped')].nunique()),
    )\
.sort_values(by=['yyyymmdd', 'pageVariant'])\
.reset_index()

## User Define Function

In [ ]:
df_data.head(2)

In [ ]:
df_data.modified_order_status.unique()

In [ ]:
df_data.pagename.unique()

In [ ]:
df_data.screen_slot_name.unique()

In [ ]:
df_data.slotname.unique()

In [ ]:
df_data.eventName.unique()

In [ ]:
agg_functions = {
    'customers': ('customer_id', 'nunique'),
    'gross_orders': ('order_id', 'nunique'),
    'net_orders': ('order_id', lambda x: x[df_data['order_status'] == 'dropped'].nunique()),
    'cobrm': ('order_id', lambda x: x[df_data['modified_order_status'] == 'COBRM'].nunique()),
    'cobra': ('order_id', lambda x: x[df_data['modified_order_status'] == 'COBRA'].nunique()),
    'ocara': ('order_id', lambda x: x[df_data['modified_order_status'] == 'OCARA'].nunique()),
    'expired': ('order_id', lambda x: x[df_data['modified_order_status'] == 'expired'].nunique()),
    'aborted': ('order_id', lambda x: x[df_data['modified_order_status'] == 'aborted'].nunique()),
    
    'ttc': ('time_spent_in_sec', 'median'),
    'cobrm_time': ('time_spent_in_sec', lambda x: x[df_data.loc[x.index, 'modified_order_status'] == 'COBRM'].median()),
    'cobra_time': ('time_spent_in_sec', lambda x: x[df_data.loc[x.index, 'modified_order_status'] == 'COBRA'].median()),
    'ocara_time': ('time_spent_in_sec', lambda x: x[df_data.loc[x.index, 'modified_order_status'] == 'OCARA'].median()),
    
    'aborted': ('order_id', lambda x: x[(df_data.loc[x.index, 'modified_order_status'] == 'aborted') & 
                                         (df_data.loc[x.index, 'pagename'] == 'aborted')].nunique()),
    
    'css_request': ('unique_event_id', 
                    lambda x: x[(df_data.loc[x.index, 'pagename'] == 'CaptainSearchScreen') 
                                & 
                                (df_data.loc[x.index, 'eventName'] == 'Ad_Request')
                               ].nunique()),
    'css_response': ('unique_event_id', 
                     lambda x: x[(df_data.loc[x.index, 'pagename'] == 'CaptainSearchScreen') 
                                 & 
                                 (df_data.loc[x.index, 'eventName'] == 'Ad_Response')
                                ].nunique()),
    'css_viewable_impression': ('unique_event_id', 
                                lambda x: x[(df_data.loc[x.index, 'pagename'] == 'CaptainSearchScreen') 
                                            & 
                                            (df_data.loc[x.index, 'eventName'] == 'Ad_Viewable_Impression')
                                           ].nunique()),
    'css_click': ('unique_event_id', 
                  lambda x: x[(df_data.loc[x.index, 'pagename'] == 'CaptainSearchScreen') 
                              & 
                              (df_data.loc[x.index, 'eventName'] == 'Ad_Click')
                             ].nunique()),
    
    'pos_request': ('unique_event_id', 
                    lambda x: x[(df_data.loc[x.index, 'pagename'] == 'PostOrderScreen') 
                                & 
                                (df_data.loc[x.index, 'eventName'] == 'Ad_Request')
                               ].nunique()),
    'pos_response': ('unique_event_id', 
                     lambda x: x[(df_data.loc[x.index, 'pagename'] == 'PostOrderScreen') 
                                 & 
                                 (df_data.loc[x.index, 'eventName'] == 'Ad_Response')
                                ].nunique()),
    'pos_viewable_impression': ('unique_event_id', 
                                lambda x: x[(df_data.loc[x.index, 'pagename'] == 'PostOrderScreen') 
                                            & 
                                            (df_data.loc[x.index, 'eventName'] == 'Ad_Viewable_Impression')
                                           ].nunique()),
    'pos_click': ('unique_event_id', 
                  lambda x: x[(df_data.loc[x.index, 'pagename'] == 'PostOrderScreen') 
                              & 
                              (df_data.loc[x.index, 'eventName'] == 'Ad_Click')
                             ].nunique()),
    
    'onTheWay_request': ('unique_event_id', 
                         lambda x: x[(df_data.loc[x.index, 'screen_slot_name'] == 'onTheWay') 
                                     & 
                                     (df_data.loc[x.index, 'eventName'] == 'Ad_Request')
                                    ].nunique()),
    'onTheWay_response': ('unique_event_id', 
                          lambda x: x[(df_data.loc[x.index, 'screen_slot_name'] == 'onTheWay') 
                                      & 
                                      (df_data.loc[x.index, 'eventName'] == 'Ad_Response')
                                     ].nunique()),
    'onTheWay_viewable_impression': ('unique_event_id', 
                                     lambda x: x[(df_data.loc[x.index, 'screen_slot_name'] == 'onTheWay') 
                                                 & 
                                                 (df_data.loc[x.index, 'eventName'] == 'Ad_Viewable_Impression')
                                                ].nunique()),
    'onTheWay_click': ('unique_event_id', 
                       lambda x: x[(df_data.loc[x.index, 'screen_slot_name'] == 'onTheWay') 
                                   & 
                                   (df_data.loc[x.index, 'eventName'] == 'Ad_Click')
                                  ].nunique()),
    
    'arrived_request': ('unique_event_id', 
                         lambda x: x[(df_data.loc[x.index, 'screen_slot_name'] == 'arrived') 
                                     & 
                                     (df_data.loc[x.index, 'eventName'] == 'Ad_Request')
                                    ].nunique()),
    'arrived_response': ('unique_event_id', 
                          lambda x: x[(df_data.loc[x.index, 'screen_slot_name'] == 'arrived') 
                                      & 
                                      (df_data.loc[x.index, 'eventName'] == 'Ad_Response')
                                     ].nunique()),
    'arrived_viewable_impression': ('unique_event_id', 
                                     lambda x: x[(df_data.loc[x.index, 'screen_slot_name'] == 'arrived') 
                                                 & 
                                                 (df_data.loc[x.index, 'eventName'] == 'Ad_Viewable_Impression')
                                                ].nunique()),
    'arrived_click': ('unique_event_id', 
                       lambda x: x[(df_data.loc[x.index, 'screen_slot_name'] == 'arrived') 
                                   & 
                                   (df_data.loc[x.index, 'eventName'] == 'Ad_Click')
                                  ].nunique()),
    
    'started_request': ('unique_event_id', 
                         lambda x: x[(df_data.loc[x.index, 'screen_slot_name'] == 'started') 
                                     & 
                                     (df_data.loc[x.index, 'eventName'] == 'Ad_Request')
                                    ].nunique()),
    'started_response': ('unique_event_id', 
                          lambda x: x[(df_data.loc[x.index, 'screen_slot_name'] == 'started') 
                                      & 
                                      (df_data.loc[x.index, 'eventName'] == 'Ad_Response')
                                     ].nunique()),
    'started_viewable_impression': ('unique_event_id', 
                                     lambda x: x[(df_data.loc[x.index, 'screen_slot_name'] == 'started') 
                                                 & 
                                                 (df_data.loc[x.index, 'eventName'] == 'Ad_Viewable_Impression')
                                                ].nunique()),
    'started_click': ('unique_event_id', 
                       lambda x: x[(df_data.loc[x.index, 'screen_slot_name'] == 'started') 
                                   & 
                                   (df_data.loc[x.index, 'eventName'] == 'Ad_Click')
                                  ].nunique()),
    
    'PostOrderOnTheWay1_request': ('unique_event_id', 
                                     lambda x: x[(df_data.loc[x.index, 'slotname'] == 'PostOrderOnTheWay1') 
                                                 & 
                                                 (df_data.loc[x.index, 'eventName'] == 'Ad_Request')
                                                ].nunique()),
    'PostOrderOnTheWay1_response': ('unique_event_id', 
                                      lambda x: x[(df_data.loc[x.index, 'slotname'] == 'PostOrderOnTheWay1') 
                                                  & 
                                                  (df_data.loc[x.index, 'eventName'] == 'Ad_Response')
                                                 ].nunique()),
    'PostOrderOnTheWay1_viewable_impression': ('unique_event_id', 
                                                 lambda x: x[(df_data.loc[x.index, 'slotname'] == 'PostOrderOnTheWay1') 
                                                             & 
                                                             (df_data.loc[x.index, 'eventName'] == 'Ad_Viewable_Impression')
                                                            ].nunique()),
    'PostOrderOnTheWay1_click': ('unique_event_id', 
                                   lambda x: x[(df_data.loc[x.index, 'slotname'] == 'PostOrderOnTheWay1') 
                                               & 
                                               (df_data.loc[x.index, 'eventName'] == 'Ad_Click')
                                              ].nunique()),
    
    'PostOrderOnTheWay2_request': ('unique_event_id', 
                                     lambda x: x[(df_data.loc[x.index, 'slotname'] == 'PostOrderOnTheWay2') 
                                                 & 
                                                 (df_data.loc[x.index, 'eventName'] == 'Ad_Request')
                                                ].nunique()),
    'PostOrderOnTheWay2_response': ('unique_event_id', 
                                      lambda x: x[(df_data.loc[x.index, 'slotname'] == 'PostOrderOnTheWay2') 
                                                  & 
                                                  (df_data.loc[x.index, 'eventName'] == 'Ad_Response')
                                                 ].nunique()),
    'PostOrderOnTheWay2_viewable_impression': ('unique_event_id', 
                                                 lambda x: x[(df_data.loc[x.index, 'slotname'] == 'PostOrderOnTheWay2') 
                                                             & 
                                                             (df_data.loc[x.index, 'eventName'] == 'Ad_Viewable_Impression')
                                                            ].nunique()),
    'PostOrderOnTheWay2_click': ('unique_event_id', 
                                   lambda x: x[(df_data.loc[x.index, 'slotname'] == 'PostOrderOnTheWay2') 
                                               & 
                                               (df_data.loc[x.index, 'eventName'] == 'Ad_Click')
                                              ].nunique()),
    
    'PostOrderOnTheWay3_request': ('unique_event_id', 
                                     lambda x: x[(df_data.loc[x.index, 'slotname'] == 'PostOrderOnTheWay3') 
                                                 & 
                                                 (df_data.loc[x.index, 'eventName'] == 'Ad_Request')
                                                ].nunique()),
    'PostOrderOnTheWay3_response': ('unique_event_id', 
                                      lambda x: x[(df_data.loc[x.index, 'slotname'] == 'PostOrderOnTheWay3') 
                                                  & 
                                                  (df_data.loc[x.index, 'eventName'] == 'Ad_Response')
                                                 ].nunique()),
    'PostOrderOnTheWay3_viewable_impression': ('unique_event_id', 
                                                 lambda x: x[(df_data.loc[x.index, 'slotname'] == 'PostOrderOnTheWay3') 
                                                             & 
                                                             (df_data.loc[x.index, 'eventName'] == 'Ad_Viewable_Impression')
                                                            ].nunique()),
    'PostOrderOnTheWay3_click': ('unique_event_id', 
                                   lambda x: x[(df_data.loc[x.index, 'slotname'] == 'PostOrderOnTheWay3') 
                                               & 
                                               (df_data.loc[x.index, 'eventName'] == 'Ad_Click')
                                              ].nunique()),
    
    'PostOrderArrived1_request': ('unique_event_id', 
                                     lambda x: x[(df_data.loc[x.index, 'slotname'] == 'PostOrderArrived1') 
                                                 & 
                                                 (df_data.loc[x.index, 'eventName'] == 'Ad_Request')
                                                ].nunique()),
    'PostOrderArrived1_response': ('unique_event_id', 
                                      lambda x: x[(df_data.loc[x.index, 'slotname'] == 'PostOrderArrived1') 
                                                  & 
                                                  (df_data.loc[x.index, 'eventName'] == 'Ad_Response')
                                                 ].nunique()),
    'PostOrderArrived1_viewable_impression': ('unique_event_id', 
                                                 lambda x: x[(df_data.loc[x.index, 'slotname'] == 'PostOrderArrived1') 
                                                             & 
                                                             (df_data.loc[x.index, 'eventName'] == 'Ad_Viewable_Impression')
                                                            ].nunique()),
    'PostOrderArrived1_click': ('unique_event_id', 
                                   lambda x: x[(df_data.loc[x.index, 'slotname'] == 'PostOrderArrived1') 
                                               & 
                                               (df_data.loc[x.index, 'eventName'] == 'Ad_Click')
                                              ].nunique()),
    
    'PostOrderArrived2_request': ('unique_event_id', 
                                     lambda x: x[(df_data.loc[x.index, 'slotname'] == 'PostOrderArrived2') 
                                                 & 
                                                 (df_data.loc[x.index, 'eventName'] == 'Ad_Request')
                                                ].nunique()),
    'PostOrderArrived2_response': ('unique_event_id', 
                                      lambda x: x[(df_data.loc[x.index, 'slotname'] == 'PostOrderArrived2') 
                                                  & 
                                                  (df_data.loc[x.index, 'eventName'] == 'Ad_Response')
                                                 ].nunique()),
    'PostOrderArrived2_viewable_impression': ('unique_event_id', 
                                                 lambda x: x[(df_data.loc[x.index, 'slotname'] == 'PostOrderArrived2') 
                                                             & 
                                                             (df_data.loc[x.index, 'eventName'] == 'Ad_Viewable_Impression')
                                                            ].nunique()),
    'PostOrderArrived2_click': ('unique_event_id', 
                                   lambda x: x[(df_data.loc[x.index, 'slotname'] == 'PostOrderArrived2') 
                                               & 
                                               (df_data.loc[x.index, 'eventName'] == 'Ad_Click')
                                              ].nunique()),
    
    'PostOrderArrived3_request': ('unique_event_id', 
                                     lambda x: x[(df_data.loc[x.index, 'slotname'] == 'PostOrderArrived3') 
                                                 & 
                                                 (df_data.loc[x.index, 'eventName'] == 'Ad_Request')
                                                ].nunique()),
    'PostOrderArrived3_response': ('unique_event_id', 
                                      lambda x: x[(df_data.loc[x.index, 'slotname'] == 'PostOrderArrived3') 
                                                  & 
                                                  (df_data.loc[x.index, 'eventName'] == 'Ad_Response')
                                                 ].nunique()),
    'PostOrderArrived3_viewable_impression': ('unique_event_id', 
                                                 lambda x: x[(df_data.loc[x.index, 'slotname'] == 'PostOrderArrived3') 
                                                             & 
                                                             (df_data.loc[x.index, 'eventName'] == 'Ad_Viewable_Impression')
                                                            ].nunique()),
    'PostOrderArrived3_click': ('unique_event_id', 
                                   lambda x: x[(df_data.loc[x.index, 'slotname'] == 'PostOrderArrived3') 
                                               & 
                                               (df_data.loc[x.index, 'eventName'] == 'Ad_Click')
                                              ].nunique()),
    
    'PostOrderStarted1_request': ('unique_event_id', 
                                     lambda x: x[(df_data.loc[x.index, 'slotname'] == 'PostOrderStarted1') 
                                                 & 
                                                 (df_data.loc[x.index, 'eventName'] == 'Ad_Request')
                                                ].nunique()),
    'PostOrderStarted1_response': ('unique_event_id', 
                                      lambda x: x[(df_data.loc[x.index, 'slotname'] == 'PostOrderStarted1') 
                                                  & 
                                                  (df_data.loc[x.index, 'eventName'] == 'Ad_Response')
                                                 ].nunique()),
    'PostOrderStarted1_viewable_impression': ('unique_event_id', 
                                                 lambda x: x[(df_data.loc[x.index, 'slotname'] == 'PostOrderStarted1') 
                                                             & 
                                                             (df_data.loc[x.index, 'eventName'] == 'Ad_Viewable_Impression')
                                                            ].nunique()),
    'PostOrderStarted1_click': ('unique_event_id', 
                                   lambda x: x[(df_data.loc[x.index, 'slotname'] == 'PostOrderStarted1') 
                                               & 
                                               (df_data.loc[x.index, 'eventName'] == 'Ad_Click')
                                              ].nunique()),
    
    'PostOrderStarted2_request': ('unique_event_id', 
                                     lambda x: x[(df_data.loc[x.index, 'slotname'] == 'PostOrderStarted2') 
                                                 & 
                                                 (df_data.loc[x.index, 'eventName'] == 'Ad_Request')
                                                ].nunique()),
    'PostOrderStarted2_response': ('unique_event_id', 
                                      lambda x: x[(df_data.loc[x.index, 'slotname'] == 'PostOrderStarted2') 
                                                  & 
                                                  (df_data.loc[x.index, 'eventName'] == 'Ad_Response')
                                                 ].nunique()),
    'PostOrderStarted2_viewable_impression': ('unique_event_id', 
                                                 lambda x: x[(df_data.loc[x.index, 'slotname'] == 'PostOrderStarted2') 
                                                             & 
                                                             (df_data.loc[x.index, 'eventName'] == 'Ad_Viewable_Impression')
                                                            ].nunique()),
    'PostOrderStarted2_click': ('unique_event_id', 
                                   lambda x: x[(df_data.loc[x.index, 'slotname'] == 'PostOrderStarted2') 
                                               & 
                                               (df_data.loc[x.index, 'eventName'] == 'Ad_Click')
                                              ].nunique()),
    
    'PostOrderStarted3_request': ('unique_event_id', 
                                     lambda x: x[(df_data.loc[x.index, 'slotname'] == 'PostOrderStarted3') 
                                                 & 
                                                 (df_data.loc[x.index, 'eventName'] == 'Ad_Request')
                                                ].nunique()),
    'PostOrderStarted3_response': ('unique_event_id', 
                                      lambda x: x[(df_data.loc[x.index, 'slotname'] == 'PostOrderStarted3') 
                                                  & 
                                                  (df_data.loc[x.index, 'eventName'] == 'Ad_Response')
                                                 ].nunique()),
    'PostOrderStarted3_viewable_impression': ('unique_event_id', 
                                                 lambda x: x[(df_data.loc[x.index, 'slotname'] == 'PostOrderStarted3') 
                                                             & 
                                                             (df_data.loc[x.index, 'eventName'] == 'Ad_Viewable_Impression')
                                                            ].nunique()),
    'PostOrderStarted3_click': ('unique_event_id', 
                                   lambda x: x[(df_data.loc[x.index, 'slotname'] == 'PostOrderStarted3') 
                                               & 
                                               (df_data.loc[x.index, 'eventName'] == 'Ad_Click')
                                              ].nunique()),    
    
} 

In [ ]:
def calculate_percentage(df, numerator, denominator):
    percentage = (df[numerator] * 100.00 / df[denominator]).round(2)
    return percentage

In [ ]:
def calculate_percentage_columns(df):
    
    df['g2n'] = calculate_percentage(df, 'net_orders', 'gross_orders')
    df['% cobrm'] = calculate_percentage(df, 'cobrm', 'gross_orders')
    df['% cobra'] = calculate_percentage(df, 'cobra', 'gross_orders')
    df['% ocara'] = calculate_percentage(df, 'ocara', 'gross_orders')
    df['% expired'] = calculate_percentage(df, 'expired', 'gross_orders')
    df['% aborted'] = calculate_percentage(df, 'aborted', 'gross_orders')
    
    ## Page group
    df['% CSS response'] = calculate_percentage(df, 'css_response', 'css_request')
    df['% CSS viewability'] = calculate_percentage(df, 'css_viewable_impression', 'css_response')
    df['% CSS CTR'] = calculate_percentage(df, 'css_click', 'css_viewable_impression')
    
    df['% POS response'] = calculate_percentage(df, 'pos_response', 'pos_request')
    df['% POS viewability'] = calculate_percentage(df, 'pos_viewable_impression', 'pos_response')
    df['% POS CTR'] = calculate_percentage(df, 'pos_click', 'pos_viewable_impression')
    
    ## Screen
    df['% onTheWay response'] = calculate_percentage(df, 'onTheWay_response', 'onTheWay_request')
    df['% onTheWay viewability'] = calculate_percentage(df, 'onTheWay_viewable_impression', 'onTheWay_response')
    df['% onTheWay CTR'] = calculate_percentage(df, 'onTheWay_click', 'onTheWay_viewable_impression')
    
    df['% arrived response'] = calculate_percentage(df, 'arrived_response', 'arrived_request')
    df['% arrived viewability'] = calculate_percentage(df, 'arrived_viewable_impression', 'arrived_response')
    df['% arrived CTR'] = calculate_percentage(df, 'arrived_click', 'arrived_viewable_impression')
    
    df['% started response'] = calculate_percentage(df, 'started_response', 'started_request')
    df['% started viewability'] = calculate_percentage(df, 'started_viewable_impression', 'started_response')
    df['% started CTR'] = calculate_percentage(df, 'started_click', 'started_viewable_impression')
    
    ## Slot
    df['% onTheWay1 response'] = calculate_percentage(df, 'PostOrderOnTheWay1_response', 'PostOrderOnTheWay1_request')
    df['% onTheWay1 viewability'] = calculate_percentage(df, 'PostOrderOnTheWay1_viewable_impression', 'PostOrderOnTheWay1_response')
    df['% onTheWay1 CTR'] = calculate_percentage(df, 'PostOrderOnTheWay1_click', 'PostOrderOnTheWay1_viewable_impression')
    
    df['% onTheWay2 response'] = calculate_percentage(df, 'PostOrderOnTheWay2_response', 'PostOrderOnTheWay2_request')
    df['% onTheWay2 viewability'] = calculate_percentage(df, 'PostOrderOnTheWay2_viewable_impression', 'PostOrderOnTheWay2_response')
    df['% onTheWay2 CTR'] = calculate_percentage(df, 'PostOrderOnTheWay2_click', 'PostOrderOnTheWay2_viewable_impression')
    
    df['% onTheWay3 response'] = calculate_percentage(df, 'PostOrderOnTheWay3_response', 'PostOrderOnTheWay3_request')
    df['% onTheWay3 viewability'] = calculate_percentage(df, 'PostOrderOnTheWay3_viewable_impression', 'PostOrderOnTheWay3_response')
    df['% onTheWay3 CTR'] = calculate_percentage(df, 'PostOrderOnTheWay3_click', 'PostOrderOnTheWay3_viewable_impression')
    
    df['% arrived1 response'] = calculate_percentage(df, 'PostOrderArrived1_response', 'PostOrderArrived1_request')
    df['% arrived1 viewability'] = calculate_percentage(df, 'PostOrderArrived1_viewable_impression', 'PostOrderArrived1_response')
    df['% arrived1 CTR'] = calculate_percentage(df, 'PostOrderArrived1_click', 'PostOrderArrived1_viewable_impression')
    
    df['% arrived2 response'] = calculate_percentage(df, 'PostOrderArrived2_response', 'PostOrderArrived2_request')
    df['% arrived2 viewability'] = calculate_percentage(df, 'PostOrderArrived2_viewable_impression', 'PostOrderArrived2_response')
    df['% arrived2 CTR'] = calculate_percentage(df, 'PostOrderArrived2_click', 'PostOrderArrived2_viewable_impression')
    
    df['% arrived3 response'] = calculate_percentage(df, 'PostOrderArrived3_response', 'PostOrderArrived3_request')
    df['% arrived3 viewability'] = calculate_percentage(df, 'PostOrderArrived3_viewable_impression', 'PostOrderArrived3_response')
    df['% arrived3 CTR'] = calculate_percentage(df, 'PostOrderArrived3_click', 'PostOrderArrived3_viewable_impression')
    
    df['% started1 response'] = calculate_percentage(df, 'PostOrderStarted1_response', 'PostOrderStarted1_request')
    df['% started1 viewability'] = calculate_percentage(df, 'PostOrderStarted1_viewable_impression', 'PostOrderStarted1_response')
    df['% started1 CTR'] = calculate_percentage(df, 'PostOrderStarted1_click', 'PostOrderStarted1_viewable_impression')
    
    df['% started2 response'] = calculate_percentage(df, 'PostOrderStarted2_response', 'PostOrderStarted2_request')
    df['% started2 viewability'] = calculate_percentage(df, 'PostOrderStarted2_viewable_impression', 'PostOrderStarted2_response')
    df['% started2 CTR'] = calculate_percentage(df, 'PostOrderStarted2_click', 'PostOrderStarted2_viewable_impression')
    
    df['% started3 response'] = calculate_percentage(df, 'PostOrderStarted3_response', 'PostOrderStarted3_request')
    df['% started3 viewability'] = calculate_percentage(df, 'PostOrderStarted3_viewable_impression', 'PostOrderStarted3_response')
    df['% started3 CTR'] = calculate_percentage(df, 'PostOrderStarted3_click', 'PostOrderStarted3_viewable_impression')
    
    
    
    df_new = df[['g2n', '% cobrm', '% cobra', '% ocara', '% expired', '% aborted',
                 'ttc', 'cobrm_time', 'cobra_time', 'ocara_time',
                 
                 '% CSS response', '% CSS viewability', '% CSS CTR',
                 '% POS response', '% POS viewability', '% POS CTR',
                 
                 '% onTheWay response', '% onTheWay viewability', '% onTheWay CTR',
                 '% arrived response', '% arrived viewability', '% arrived CTR',
                 '% started response', '% started viewability', '% started CTR',
                 
                 '% onTheWay1 response', '% onTheWay1 viewability', '% onTheWay1 CTR',
                 '% onTheWay2 response', '% onTheWay2 viewability', '% onTheWay2 CTR',
                 '% onTheWay3 response', '% onTheWay3 viewability', '% onTheWay3 CTR',
                 
                 '% arrived1 response', '% arrived1 viewability', '% arrived1 CTR',
                 '% arrived2 response', '% arrived2 viewability', '% arrived2 CTR',
                 '% arrived3 response', '% arrived3 viewability', '% arrived3 CTR',
                 
                 '% started1 response', '% started1 viewability', '% started1 CTR',
                 '% started2 response', '% started2 viewability', '% started2 CTR',
                 '% started3 response', '% started3 viewability', '% started3 CTR',
                 
                ]] 
    
    return df_new

In [ ]:
# df_city.columns

## Analysis 

In [ ]:
df_data.head(2)

In [ ]:
df_city = df_data\
.groupby(['pageVariant', 'city'])\
.agg(**agg_functions)\
.sort_values(by=['city', 'pageVariant'])\
.reset_index()

df_city

In [ ]:
df_city_with_percentage = calculate_percentage_columns(df_city)
df_city_with_percentage

In [ ]:
df_city_with_percentage.to_clipboard(index=False)

In [ ]:
df_service_level = df_data\
.groupby(['pageVariant', 'service_name'])\
.agg(**agg_functions)\
.sort_values(by=['service_name', 'pageVariant'])\
.reset_index()

df_service_level

In [ ]:
df_service_with_percentage = calculate_percentage_columns(df_service_level)
df_service_with_percentage

In [ ]:
df_service_with_percentage.to_clipboard(index=False)

In [ ]:
df_daily = df_data\
.groupby(['yyyymmdd', 'city', 'pageVariant'])\
.agg(**agg_functions)\
.sort_values(by=['yyyymmdd', 'city', 'pageVariant'])\
.reset_index()

df_daily

In [ ]:
df_daily_with_percentage = calculate_percentage_columns(df_daily)
df_daily_with_percentage

In [ ]:
df_daily_with_percentage.to_clipboard(index=False)